# PayShield AI — Explainable Risk Detection

## AI-Powered Payment Success Optimization

### Objective

Explain why PayShield predicts that a receiver bank
may experience payment failures.

The system will identify:

- Important risk factors
- Current abnormal signals
- Risk probability
- Risk level
- Human-readable explanations

The goal is to make AI predictions understandable
and actionable.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

In [2]:
df = pd.read_csv(
    "../data/processed/ml_dataset.csv"
)

print("Dataset shape:", df.shape)

Dataset shape: (50944, 22)


In [3]:
X = df.drop(columns=["risk_target"])
y = df["risk_target"]

print("Features:", X.shape)
print("Target:", y.shape)

Features: (50944, 21)
Target: (50944,)


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training:", X_train.shape)
print("Testing:", X_test.shape)

Training: (40755, 21)
Testing: (10189, 21)


In [5]:
rf_model = RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_model.fit(
    X_train,
    y_train
)

print("Random Forest trained successfully!")

Random Forest trained successfully!


In [6]:
risk_probability = rf_model.predict_proba(
    X_test
)[:, 1]

print("First 10 risk probabilities:")
print(risk_probability[:10])

First 10 risk probabilities:
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


In [7]:
results = X_test.copy()

results["actual_risk"] = y_test.values

results["risk_probability"] = risk_probability

results = results.reset_index(drop=True)

results.head()

,transaction_count,failure_rate,timeout_rate,avg_latency,max_latency,p95_latency,bank_error_rate,avg_amount,max_amount,hour,...,previous_latency,previous_timeout_rate,failure_rate_change,latency_change,timeout_rate_change,rolling_failure_rate,rolling_latency,rolling_timeout_rate,actual_risk,risk_probability
0,4,0.0,0.0,1064.240,1861.51,1748.2105,0.0,204.7275,309.03,20,...,1908.440000,0.0,0.0,-844.200000,0.0,0.0,1326.718333,0.0,0,0.0
1,2,0.0,0.0,936.830,1294.56,1258.7870,0.0,427.0750,710.07,17,...,1110.952500,0.0,0.0,-174.122500,0.0,0.0,1301.178611,0.0,0,0.0
2,2,0.0,0.0,965.595,1156.23,1137.1665,0.0,981.7650,1404.47,15,...,1323.555000,0.0,0.0,-357.960000,0.0,0.0,1148.745000,0.0,0,0.0
3,2,0.0,0.0,1571.595,2815.04,2690.6955,0.0,211.0400,340.84,8,...,1435.096667,0.0,0.0,136.498333,0.0,0.0,1317.770556,0.0,0,0.0
4,1,0.0,0.0,903.040,903.04,903.0400,0.0,2002.6800,2002.68,2,...,412.270000,0.0,0.0,490.770000,0.0,0.0,744.826667,0.0,0,0.0


In [8]:
feature_importance = pd.DataFrame({
    "feature": X.columns,
    "importance": rf_model.feature_importances_
})

feature_importance = (
    feature_importance
    .sort_values(
        "importance",
        ascending=False
    )
    .reset_index(drop=True)
)

feature_importance

,feature,importance
0,rolling_latency,0.250088
1,avg_latency,0.174527
2,previous_latency,0.161687
3,p95_latency,0.112608
4,max_latency,0.111822
5,rolling_failure_rate,0.044404
6,latency_change,0.021587
7,hour,0.019586
8,rolling_timeout_rate,0.019320
9,previous_failure_rate,0.014693


In [9]:
feature_importance.head(10)

,feature,importance
0,rolling_latency,0.250088
1,avg_latency,0.174527
2,previous_latency,0.161687
3,p95_latency,0.112608
4,max_latency,0.111822
5,rolling_failure_rate,0.044404
6,latency_change,0.021587
7,hour,0.019586
8,rolling_timeout_rate,0.019320
9,previous_failure_rate,0.014693


In [10]:
high_risk_cases = results[
    results["risk_probability"] >= 0.20
].copy()

print(
    "Number of high-risk cases:",
    len(high_risk_cases)
)

Number of high-risk cases: 51


In [11]:
highest_risk = results.loc[
    results["risk_probability"].idxmax()
]

highest_risk

transaction_count           2.000000
failure_rate               50.000000
timeout_rate                0.000000
avg_latency              2962.540000
max_latency              4248.920000
p95_latency              4120.282000
bank_error_rate            50.000000
avg_amount                264.260000
max_amount                316.550000
hour                       15.000000
day_of_week                 1.000000
is_weekend                  0.000000
previous_failure_rate     100.000000
previous_latency         5457.515000
previous_timeout_rate      50.000000
failure_rate_change       -50.000000
latency_change          -2494.975000
timeout_rate_change       -50.000000
rolling_failure_rate       83.333333
rolling_latency          3764.731667
rolling_timeout_rate       50.000000
actual_risk                 1.000000
risk_probability            0.990000
Name: 4350, dtype: float64

In [13]:
def generate_explanation(row):

    explanations = []

    if row["failure_rate"] >= 10:
        explanations.append(
            "Failure rate is unusually high."
        )

    if row["timeout_rate"] >= 10:
        explanations.append(
            "Timeout rate is unusually high."
        )

    if row["avg_latency"] >= 3000:
        explanations.append(
            "Average payment latency is very high."
        )

    if row["p95_latency"] >= 5000:
        explanations.append(
            "P95 latency indicates severe slowdowns."
        )

    if row["bank_error_rate"] >= 10:
        explanations.append(
            "Bank-related errors are elevated."
        )

    if row["failure_rate_change"] >= 5:
        explanations.append(
            "Failure rate has increased significantly."
        )

    if row["latency_change"] >= 1000:
        explanations.append(
            "Payment latency has increased significantly."
        )

    if row["timeout_rate_change"] >= 5:
        explanations.append(
            "Timeout rate has increased significantly."
        )

    if row["rolling_failure_rate"] >= 5:
        explanations.append(
            "Recent failure trend is elevated."
        )

    if row["rolling_latency"] >= 3000:
        explanations.append(
            "Recent latency trend is elevated."
        )

    return explanations

In [14]:
explanation = generate_explanation(
    highest_risk
)

print(
    "Risk probability:",
    round(
        highest_risk["risk_probability"] * 100,
        2
    ),
    "%"
)

print("\nReasons:")

for reason in explanation:
    print("•", reason)

Risk probability: 99.0 %

Reasons:
• Failure rate is unusually high.
• Bank-related errors are elevated.
• Recent failure trend is elevated.
• Recent latency trend is elevated.


In [15]:
def risk_level(probability):

    if probability >= 0.80:
        return "HIGH"

    elif probability >= 0.50:
        return "MEDIUM"

    else:
        return "LOW"

In [16]:
results["risk_level"] = (
    results["risk_probability"]
    .apply(risk_level)
)

results["explanation"] = (
    results.apply(
        generate_explanation,
        axis=1
    )
)

results.head()

,transaction_count,failure_rate,timeout_rate,avg_latency,max_latency,p95_latency,bank_error_rate,avg_amount,max_amount,hour,...,failure_rate_change,latency_change,timeout_rate_change,rolling_failure_rate,rolling_latency,rolling_timeout_rate,actual_risk,risk_probability,risk_level,explanation
0,4,0.0,0.0,1064.240,1861.51,1748.2105,0.0,204.7275,309.03,20,...,0.0,-844.200000,0.0,0.0,1326.718333,0.0,0,0.0,LOW,[]
1,2,0.0,0.0,936.830,1294.56,1258.7870,0.0,427.0750,710.07,17,...,0.0,-174.122500,0.0,0.0,1301.178611,0.0,0,0.0,LOW,[]
2,2,0.0,0.0,965.595,1156.23,1137.1665,0.0,981.7650,1404.47,15,...,0.0,-357.960000,0.0,0.0,1148.745000,0.0,0,0.0,LOW,[]
3,2,0.0,0.0,1571.595,2815.04,2690.6955,0.0,211.0400,340.84,8,...,0.0,136.498333,0.0,0.0,1317.770556,0.0,0,0.0,LOW,[]
4,1,0.0,0.0,903.040,903.04,903.0400,0.0,2002.6800,2002.68,2,...,0.0,490.770000,0.0,0.0,744.826667,0.0,0,0.0,LOW,[]


In [17]:
def create_alert(row):

    if row["risk_probability"] < 0.20:

        return "NO ALERT"

    elif row["risk_probability"] < 0.50:

        return "MONITOR"

    elif row["risk_probability"] < 0.80:

        return "WARNING"

    else:

        return "HIGH RISK ALERT"

In [18]:
results["alert"] = (
    results.apply(
        create_alert,
        axis=1
    )
)

results[
    [
        "risk_probability",
        "risk_level",
        "alert"
    ]
].head(20)

,risk_probability,risk_level,alert
0,0.0,LOW,NO ALERT
1,0.0,LOW,NO ALERT
2,0.0,LOW,NO ALERT
3,0.0,LOW,NO ALERT
4,0.0,LOW,NO ALERT
5,0.0,LOW,NO ALERT
6,0.0,LOW,NO ALERT
7,0.0,LOW,NO ALERT
8,0.0,LOW,NO ALERT
9,0.0,LOW,NO ALERT


In [19]:
def payshield_prediction(row):

    probability = row["risk_probability"]

    level = risk_level(probability)

    reasons = generate_explanation(row)

    print("===================================")
    print("          PAYSHIELD AI")
    print("===================================")

    print(
        f"Risk Probability: {probability:.1%}"
    )

    print(
        f"Risk Level: {level}"
    )

    if probability >= 0.20:

        print("\n⚠️ PAYMENT RISK DETECTED")

        print("\nMain reasons:")

        if reasons:

            for reason in reasons:
                print("•", reason)

        else:

            print(
                "• Model detected a risk pattern "
                "from multiple signals."
            )

    else:

        print(
            "\n✓ No significant payment risk detected."
        )

In [20]:
payshield_prediction(
    highest_risk
)

          PAYSHIELD AI
Risk Probability: 99.0%
Risk Level: HIGH

⚠️ PAYMENT RISK DETECTED

Main reasons:
• Failure rate is unusually high.
• Bank-related errors are elevated.
• Recent failure trend is elevated.
• Recent latency trend is elevated.


In [21]:
results.to_csv(
    "../data/processed/explainable_predictions.csv",
    index=False
)

print(
    "Explainable predictions saved successfully!"
)

Explainable predictions saved successfully!
